# 3 · Reproduce the figures

The three figures in the paper, from committed results and stored predictions.

Figure 1 needs no data at all — it reads `results/results_2x2.csv`. Figures 2 and
3 need the prepared subsets and the prediction records respectively.

Command-line equivalent: `python scripts/make_figures.py --all`

In [ ]:
# --- environment ------------------------------------------------------------
# Local: nothing to do beyond `pip install -e .` in the repo root.
# Colab: uncomment, then set the two roots below.
#
# from google.colab import drive; drive.mount("/content/drive")
# !pip install -q -e "/content/drive/MyDrive/<path to repo>"

import os

# Where the SPEED+ download and the prepared subsets live.
os.environ.setdefault("SPEEDPLUS_DATA_ROOT", "/path/to/speedplusv2")
# Where checkpoints, predictions and figures go. On Colab put this on the
# mounted Drive so a disconnect does not cost the run.
os.environ.setdefault("SPEEDPLUS_ARTIFACTS", "/path/to/artifacts")

# On Colab, unpack image archives to local disk rather than reading tens of
# thousands of small files across the Drive mount. Set to None when local.
LOCAL_IMAGES = None  # e.g. "/content/images"

In [ ]:
from pathlib import Path

OUTPUT = Path("figures")
OUTPUT.mkdir(exist_ok=True)

## Figure 1 · Rotation error per arm and domain

The dashed line is the point. Read distances *from* it: an error of 2.05 rad
against a 2.207 chance level is a model that has learned almost nothing about
orientation, which the raw number hides completely.

In [ ]:
import pandas as pd

from speedplus_pose.figures import rotation_error_by_arm

RESULTS = Path("..") / "results" / "results_2x2.csv"
LABELS = {"frozen_no_aug": "frozen", "frozen_aug": "frozen+aug",
          "unfrozen_no_aug": "layer 4", "unfrozen_aug": "layer 4+aug"}

results = pd.read_csv(RESULTS)
arms = list(LABELS)

errors, stds = {}, {}
for domain in ("synthetic", "lightbox", "sunlamp"):
    grouped = results[results["domain"] == domain].groupby("arm")["rotation"]
    errors[domain] = {LABELS[a]: grouped.mean()[a] for a in arms}
    stds[domain] = {LABELS[a]: (grouped.std(ddof=1).get(a, 0.0) or 0.0) for a in arms}

rotation_error_by_arm([LABELS[a] for a in arms], errors, stds,
                      OUTPUT / "rotation_error_by_arm.png")

## Figure 2 · What the augmentation actually models

The figure that carries the argument. Left: augmented synthetic training samples.
Right: real test samples, sunlamp on top and lightbox below.

The augmented distribution resembles lightbox's diffuse light far more than
sunlamp's specular highlights, hard shadows and saturation — which is why
augmentation improves one on every seed and leaves the other inside seed noise.

In [ ]:
import numpy as np

from speedplus_pose.config import AugmentationConfig, load_config
from speedplus_pose.experiment import build_dataset
from speedplus_pose.figures import image_grid
from speedplus_pose.paths import Paths
from speedplus_pose.transforms import build_train_transform

SAMPLES = 8
paths = Paths()
config = load_config("unfrozen_aug")
rng = np.random.default_rng(0)

# preview=True keeps the stack identical but skips normalisation, so it renders.
augmented = build_dataset(
    "train", paths,
    build_train_transform(config.augmentation, config.image, preview=True),
    local_images=LOCAL_IMAGES,
)
picks = rng.choice(len(augmented), SAMPLES, replace=False)
image_grid([augmented[int(i)][0].numpy() for i in picks], 2, SAMPLES // 2,
           OUTPUT / "train_transforms_2rows.png")

In [ ]:
# Real samples: same resize and grayscale, no augmentation - like for like.
plain = build_train_transform(AugmentationConfig(enabled=False), config.image, preview=True)

real = []
for domain in ("sunlamp", "lightbox"):
    dataset = build_dataset(domain, paths, plain, local_images=LOCAL_IMAGES)
    real.extend(dataset[int(i)][0].numpy()
                for i in rng.choice(len(dataset), SAMPLES // 2, replace=False))

image_grid(real, 2, SAMPLES // 2, OUTPUT / "real_2rows.png")
print("top row: sunlamp    bottom row: lightbox")

## Figure 3 · Conformal reliability

Nominal versus measured coverage, swept across the whole range. The diagonal is a
perfectly honest model; distance below it is how much the stated confidence
overstates reality.

The title is off by default — it names the seed, and the paper's build is
anonymised.

In [ ]:
from speedplus_pose.coverage import scores_from_record, sweep_reliability
from speedplus_pose.evaluate import load_predictions
from speedplus_pose.figures import reliability

ARMS = ["unfrozen_no_aug_seed42", "unfrozen_aug_seed42"]
PLOT_DOMAINS = ("in_domain_test", "sunlamp", "lightbox")

alphas = np.round(np.arange(0.01, 0.51, 0.01), 2)
nominal = 1 - alphas

for arm in ARMS:
    records = load_predictions(paths.predictions / f"{arm}.pkl")
    scores = {s: scores_from_record(records[s]) for s in ("calibration", *PLOT_DOMAINS)}
    curves = {
        metric: sweep_reliability(scores["calibration"],
                                  {d: scores[d] for d in PLOT_DOMAINS}, metric, alphas)
        for metric in ("translation_metres", "rotation")
    }
    reliability(nominal, curves, arm, path=OUTPUT / f"reliability_{arm}.png")

`sweep_reliability` asserts every curve is non-decreasing in nominal coverage —
guaranteed by the maths, which is exactly what makes it a sharp check on the
code. If it raises, the bug is in the pipeline, not the data.